# AutoGrasp V2 — object-agnostic collection

Collects delicate, consistent grasps on **any object**. Reuses V1's hardware-proven grasp
machinery (the cell that made the 97% policy) — which is ALREADY object-agnostic: it segments
with `sam3_query(color, ACTIVE)` for any object name. V2 only swaps in the **shape-gated ladder**
for the angle, adds **delicacy/weight/deformation** gating, and records **real wrist_fz**.

**Tomorrow's run order:** ① Config → ② Imports → ③ Offline self-test (no robot) →
④ Bringup → ⑤ **Perception dry-run (no motion)** → ⑥ Collect.
Do ⑤ FIRST on each new object — confirm SAM3 sees it before anything moves.
V1 is frozen at tag `v1-pipeline`.


## 1 · Config — the ONLY object-specific cell


In [ ]:
OBJECT_NAME   = "strawberry"        # SAM3 concept prompt + GraspMemory key
TASK          = f"pick up the {OBJECT_NAME}"
PART_PROMPT   = None                # e.g. "handle" for part-directed grasping; None = whole object
DELICATE      = True                # low force seed + 1-retry (produce); False for rigid
INSTANCE_ID   = f"{OBJECT_NAME}_1"  # bump when you swap a fresh instance (degradation)

GRIPPER_MAX_MM  = 105.0
DELICATE_CAP_N  = 12.0
FRICTION_COEF   = 0.4               # bare AX-12 fingers (anti-slip floor; DeliGrasp refines live)
TARGET_EPISODES = 100
COVERAGE_HALF_M = 0.09             # ±9cm (wider than V1's ±6 — attacks the measured OOD wall)

# SAM3 PROMPT TIP (measured 2026-07-27): specific nouns segment best. "red cube" scored 0.90,
# bare "block" returned NO mask. Prefer "yellow banana","ripe strawberry","white mug" over "object".
print(f"object={OBJECT_NAME!r}  part={PART_PROMPT}  delicate={DELICATE}  target={TARGET_EPISODES}")


## 2 · Imports (V2 modules — 30 unit tests green)


In [ ]:
import sys, os, json, tempfile, base64, socket as _sock, numpy as np, cv2
sys.path.insert(0, os.path.expanduser("~/magpie_control/src"))
sys.path.insert(0, os.path.expanduser("~/magpie_control/scripts"))
from magpie_control.v2.grasp_planner    import plan, priors_from_memory
from magpie_control.v2.state_builder     import build_state, fz_is_live, WrenchCapture, STATE_NAMES
from magpie_control.v2.deformation_check import assess as deform_assess
from magpie_control.v2.weight_estimate   import assess as weigh, reconcile_force
from magpie_control.v2.episode_meta      import EpisodeMeta, PHASES
print("V2 modules loaded. state:", STATE_NAMES)


## 3 · OFFLINE SELF-TEST — no robot
Must print `READY` before hardware. Same code as `scripts/v2_selftest.py`.


In [ ]:
import importlib.util
_sp = importlib.util.spec_from_file_location("v2st", os.path.expanduser("~/magpie_control/scripts/v2_selftest.py"))
_st = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(_st)
assert _st.run(), "offline pipeline NOT ready"
print("\n^ decision pipeline sound.")


## 4 · Perception helpers — REAL SAM3 (tested on real images)
`sam3_query` is the exact V1 bridge; `best_mask` picks the top-scoring detection;
`perceive_and_plan` runs concept-prompt → mask → shape-gated planner.


In [ ]:
SAM3_SOCK = "/tmp/sam3.sock"
def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
    tmp = tempfile.mktemp(suffix=".jpg"); cv2.imwrite(tmp, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    try:
        with _sock.socket(_sock.AF_UNIX, _sock.SOCK_STREAM) as s:
            s.connect(sock_path); s.sendall((json.dumps({"image": tmp, "query": query})+"\n").encode())
            raw=b""
            while True:
                c=s.recv(65536)
                if not c: break
                raw+=c
        d=json.loads(raw.decode().strip())
        if "error" in d: raise RuntimeError(d["error"])
        boxes=np.array(d["boxes"],float); scores=np.array(d["scores"],float); mask=None
        if d.get("mask_b64") and len(boxes)>0:
            h,w=d["mask_shape"]
            mask=np.frombuffer(base64.b64decode(d["mask_b64"]),np.uint8).reshape(h,w).astype(bool)
        return boxes, scores, mask
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

def best_mask(img_rgb, prompt):
    boxes, scores, mask = sam3_query(img_rgb, prompt)
    if mask is None or mask.sum() < 200:
        return None, 0.0
    return mask.astype(np.uint8)*255, float(scores.max() if len(scores) else 0.0)

def depth_scale_mm_per_px(depth, mask, fx):
    """mm-per-pixel at the object plane from median depth under the mask."""
    z = depth[mask.astype(bool)].astype(float); z = z[z>0]
    if z.size < 20: return None
    return float(np.median(z) / fx)      # depth(mm) / focal(px)

def perceive_and_plan(node, gm):
    prompt = PART_PROMPT or OBJECT_NAME
    mask, score = best_mask(node.color, prompt)
    if mask is None:
        print(f"  SAM3 found no '{prompt}' — reword the prompt (specific nouns work best)"); return None
    fx = node.caminfo.k[0]
    mpp = depth_scale_mm_per_px(node.depth, mask, fx)
    if mpp is None:
        print("  depth too sparse under mask — nudge closer / re-scan"); return None
    fpri, wpri = priors_from_memory(gm, OBJECT_NAME)
    p = plan(mask, mpp, object_name=OBJECT_NAME, force_prior_n=fpri, width_prior_mm=wpri,
             max_width_mm=GRIPPER_MAX_MM, delicate=DELICATE)
    print(f"  SAM3 score={score:.2f} | [{p.method}] yaw={p.grasp_yaw_deg:.0f} "
          f"width={p.width_mm}mm seed={p.seed_force_n}N band={p.aperture_band_mm} prior={p.have_prior}")
    return p, mask, mpp


## 5 · PERCEPTION DRY-RUN — no motion  ⚠️ camera only, arm does NOT move
**Run this first on every new object.** Grabs one camera frame, segments your object, overlays
the planned grasp. If the overlay looks right, proceed to collect. If not, reword `OBJECT_NAME`.


In [ ]:
def perception_dryrun(node, gm):
    node.spin(6)
    out = perceive_and_plan(node, gm)
    if out is None:
        print("  DRY-RUN: no plan — fix perception before collecting"); return
    p, mask, mpp = out
    vis = node.color.copy()
    cnts,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cnts, -1, (0,255,0), 2)
    cx,cy = map(int, p.center_px); a=np.radians(p.grasp_yaw_deg); L=40
    cv2.line(vis,(int(cx-L*np.cos(a)),int(cy-L*np.sin(a))),(int(cx+L*np.cos(a)),int(cy+L*np.sin(a))),(255,0,0),3)
    cv2.circle(vis,(cx,cy),6,(0,0,255),-1)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,5)); plt.imshow(vis); plt.title(f"{OBJECT_NAME}: {p.method} yaw={p.grasp_yaw_deg:.0f} seed={p.seed_force_n}N"); plt.axis("off"); plt.show()
    print("  green=mask  blue=grasp line  red=grasp center. Looks right? -> collect.")
# perception_dryrun(node, gm)


## 6 · Robot bringup  ⚠️ hardware
Terminal: `bash ~/magpie_control/scripts/bringup_gripper_ft.sh`. Then build the V1 collect `node`
(v1_collect.ipynb cells c01→c04→a8581dd2) and attach the FT capture + memory:


In [ ]:
import rclpy
from geometry_msgs.msg import WrenchStamped
from grasp_memory import GraspMemory
# node = <build exactly as v1_collect cells c01,c03,c04 do — drivers, cameras, services>
# node.ft = WrenchCapture(node, WrenchStamped)          # <-- REAL wrist_fz (fixes the V1 all-zero)
# node.spin(10); assert node.ft.has_reading, "FT sensor silent — check ft_sensor_node"
gm = GraspMemory(os.path.expanduser("~/magpie_control/data/grasp_log"))
print("node up. Run cell 5 (dry-run) on your object, THEN cell 7.")


## 7 · Collect  ⚠️ hardware — reuses V1's proven grasp cell

The safest path: run V1 collect cell `17ea9af9` (the hardware-proven grasp/lift/place machinery
that made the 97% policy) with two injections. It is **already object-agnostic** — it segments
with `sam3_query(color, ACTIVE)`. So:

1. **Object** — set `ACTIVE = OBJECT_NAME` (instead of `ACTIVE_OVERRIDE='red block'`).
2. **Angle** — replace its minAreaRect block (`_flat_mask = minAreaRect(...)`, ~line 945-965)
   with the ladder: `grasp_yaw = plan(mask_use, mpp, object_name=ACTIVE, ...).grasp_yaw_deg`.
3. **Gates** — after lift, add the V2 checks before commit.


In [ ]:
# The V2 decision wrapper the V1 grasp cell calls at its gate point.
# (Paste this into cell 17ea9af9 at the reward-gate, replacing the held-only check.)
def v2_gate(node, plan_p, gm, fz_baseline):
    seated_ap = node.gs.position
    # weight from the real wrist_fz shift
    w = weigh(fz_baseline, node.ft.fz, delicate_force_cap_n=DELICATE_CAP_N)
    force, safe = reconcile_force(node.gs.force, w, DELICATE_CAP_N)
    if not safe:
        print(f"  [weight] {w.note} -> {force}N clamp; may slip (logged)")
    held = plan_p.aperture_band_mm[0] <= seated_ap <= plan_p.aperture_band_mm[1]
    expected_w = plan_p.expected_width_mm if plan_p.have_prior else 0.0   # cold-start=force-only
    # deformation target = the weight-RECONCILED force (what DeliGrasp legitimately
    # ramped to for THIS object's weight), NOT the raw seed. Using the seed would
    # false-flag any object heavier than its seed as 'crushed' (caught on the apple).
    deform = deform_assess(expected_w, seated_ap,
                           applied_force_n=node.gs.force, target_force_n=max(force, plan_p.seed_force_n))
    print(f"  held={held} gentle={deform.gentleness} ({deform.reason}) mass={w.mass_g}g")
    return held, deform, w

# angle injection helper (replaces minAreaRect in the V1 cell):
def v2_grasp_yaw(mask_use, mpp):
    from magpie_control.v2.grasp_planner import plan as _plan
    return _plan(mask_use.astype(np.uint8)*255, mpp, object_name=OBJECT_NAME,
                 max_width_mm=GRIPPER_MAX_MM, delicate=DELICATE)

print("v2_gate + v2_grasp_yaw defined. Inject into V1 cell 17ea9af9 per the steps above.")
print("Full keep rule:  keep = held and gemini_grade>=0.6 and deform.ok")
print("Remember: node.ft.fz must be LIVE (fz_is_live guard) or the wrist_fz bug recurs.")


## Notes for tomorrow
- **Per new object:** edit cell 1 (`OBJECT_NAME`, `DELICATE`), run cell 5 dry-run FIRST, then collect.
- **Prompt wording matters** (measured): use specific nouns. Bad prompt = no mask = no grasp.
- **Delicacy is software** on the bare grippers: 1.5N seed, DeliGrasp ramps, cap 12N, deformation gate.
- **Weight** is measured per lift from wrist_fz; heavy+delicate conflicts are flagged, not crushed.
- **Why reuse the V1 cell** rather than a fresh loop: its motion/DeliGrasp/lift/place code is
  hardware-proven (97% policy). V2 changes only the angle source and the gate — lowest-risk path.
